# 02 - Heart Disease Risk Model Development
## Coronary Artery Disease (CAD) Classification (UCI Dataset)

> **Clinical Decision-Support Prototype Note**:
> This notebook documents the complete model development lifecycle for the UCI Heart Disease prediction task:
> 1. Target definition and clinical task framing (Angiographic CAD presence: `num > 0`).
> 2. Leakage-free preprocessing and clinical feature engineering.
> 3. Comparative evaluation of baseline and candidate classifiers on the validation fold.
> 4. Final unbiased evaluation on held-out test data.
> 5. ROC / Precision-Recall curve visualization and confusion matrix analysis.
> 6. Discussion of sample size, class balance, and clinical limitations.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

PROJECT_ROOT = Path('.').resolve().parent.parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Machine_Learning.src.data_loader import load_heart_disease_data
from Machine_Learning.src.preprocessing import (
    prepare_heart_disease_splits,
    preprocess_inference_input,
    HEART_DISEASE_RAW_FEATURES,
)
from Machine_Learning.src.train_heart_disease import get_candidate_models, train_and_select_best_model
from Machine_Learning.src.evaluate import evaluate_binary_classifier, print_evaluation_summary

sns.set_theme(style='whitegrid', palette='muted')
print('Environment initialized successfully. Project root:', PROJECT_ROOT)


Environment initialized successfully. Project root: C:\Users\lodha\Downloads\Heart_Disease_Screening


## 1. Data Ingestion & Target Definition Verification
The target `num` represents angiographic coronary stenosis severity:
- `num == 0`: Absence of angiographic CAD (<50% diameter stenosis in major vessels).
- `num >= 1`: Presence of angiographic CAD (>=50% stenosis across 1 to 4 vessels).
Here we train binary screening classifiers to predict CAD presence (`CAD_Status = (num > 0).astype(int)`).


In [2]:
df_raw = load_heart_disease_data()
splits = prepare_heart_disease_splits(
    df_raw, test_size=0.2, val_size=0.1, random_state=42, target_mode='binary'
)

print(f'Total Dataset Records: {len(df_raw)}')
print(f'Training fold       : {splits.X_train.shape[0]} samples ({round(splits.X_train.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Validation fold     : {splits.X_val.shape[0]} samples ({round(splits.X_val.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Held-out Test fold  : {splits.X_test.shape[0]} samples ({round(splits.X_test.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Transformed Features: {splits.X_train.shape[1]}')
print(f'CAD Positive Prevalence in Train: {np.mean(splits.y_train):.3f}, Test: {np.mean(splits.y_test):.3f}')


2026-10-15 00:02:55,711 - Machine_Learning.src.data_loader - INFO - Loaded Heart Disease UCI dataset: shape=(920, 16) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_disease_uci.csv


Total Dataset Records: 920
Training fold       : 644 samples (70.0%)
Validation fold     : 92 samples (10.0%)
Held-out Test fold  : 184 samples (20.0%)
Transformed Features: 29
CAD Positive Prevalence in Train: 0.553, Test: 0.554


## 2. Baseline & Candidate Model Comparison
We evaluate five diverse classifiers on the validation fold without touching the held-out test set:
1. **Dummy (Stratified)**: Random chance baseline matching class prior.
2. **Logistic Regression (Balanced)**: Linear baseline with L2 regularization and inverse class frequency weighting.
3. **Random Forest**: Non-linear ensemble (150 trees, max depth 6).
4. **Gradient Boosting**: Sequential boosting model (120 estimators, learning rate 0.05).
5. **Support Vector Classifier (RBF)**: Non-linear kernel boundary with probability estimates.


In [3]:
best_name, best_model, val_results = train_and_select_best_model(splits, random_state=42)

val_summary = []
for name, res in val_results.items():
    val_summary.append({
        'Model': name,
        'Val Accuracy': res['accuracy'],
        'Val Balanced Acc': res['balanced_accuracy'],
        'Val Sensitivity (Recall)': res['recall_sensitivity'],
        'Val Specificity': res['specificity'],
        'Val Precision': res['precision'],
        'Val F1': res['f1_score'],
        'Val ROC-AUC': res.get('roc_auc'),
        'Val Brier': res.get('brier_score')
    })

val_df = pd.DataFrame(val_summary).sort_values(by='Val ROC-AUC', ascending=False)
print('Candidate Model Validation Comparison:')
print(val_df.to_string(index=False))


2026-10-15 00:02:56,018 - Machine_Learning.src.train_heart_disease - INFO - Evaluating 5 candidate models on validation fold...


2026-10-15 00:02:56,049 - Machine_Learning.src.train_heart_disease - INFO - Candidate [Dummy (Stratified Baseline)]: Val ROC-AUC=0.4794, Val F1=0.5524, Val Balanced Acc=0.4794


2026-10-15 00:02:56,122 - Machine_Learning.src.train_heart_disease - INFO - Candidate [Logistic Regression (Balanced)]: Val ROC-AUC=0.8360, Val F1=0.8077, Val Balanced Acc=0.7776


2026-10-15 00:02:56,704 - Machine_Learning.src.train_heart_disease - INFO - Candidate [Random Forest (Depth 6)]: Val ROC-AUC=0.8312, Val F1=0.7961, Val Balanced Acc=0.7678


2026-10-15 00:02:57,188 - Machine_Learning.src.train_heart_disease - INFO - Candidate [Gradient Boosting]: Val ROC-AUC=0.8379, Val F1=0.8113, Val Balanced Acc=0.7752


C:\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
2026-10-15 00:02:57,348 - Machine_Learning.src.train_heart_disease - INFO - Candidate [Support Vector Classifier (RBF)]: Val ROC-AUC=0.8460, Val F1=0.8190, Val Balanced Acc=0.7874


2026-10-15 00:02:57,349 - Machine_Learning.src.train_heart_disease - INFO - Selected top model: 'Support Vector Classifier (RBF)' (Validation Score: 0.8460)


Candidate Model Validation Comparison:
                          Model  Val Accuracy  Val Balanced Acc  Val Sensitivity (Recall)  Val Specificity  Val Precision  Val F1  Val ROC-AUC  Val Brier
Support Vector Classifier (RBF)        0.7935            0.7874                    0.8431           0.7317         0.7963  0.8190       0.8460     0.1594
              Gradient Boosting        0.7826            0.7752                    0.8431           0.7073         0.7818  0.8113       0.8379     0.1625
 Logistic Regression (Balanced)        0.7826            0.7776                    0.8235           0.7317         0.7925  0.8077       0.8360     0.1702
        Random Forest (Depth 6)        0.7717            0.7678                    0.8039           0.7317         0.7885  0.7961       0.8312     0.1637
    Dummy (Stratified Baseline)        0.4891            0.4794                    0.5686           0.3902         0.5370  0.5524       0.4794     0.5109


## 3. Final Evaluation on Held-Out Test Data
The top-performing model selected from validation is evaluated on the completely unseen Test fold (184 patients).


In [4]:
y_test_pred = best_model.predict(splits.X_test)
y_test_prob = best_model.predict_proba(splits.X_test)[:, 1]

test_metrics = evaluate_binary_classifier(splits.y_test, y_test_pred, y_test_prob)
print_evaluation_summary(test_metrics, f'HELD-OUT TEST SET ({best_name})')



EVALUATION SUMMARY: HELD-OUT TEST SET (Support Vector Classifier (RBF))
Sample Count       : 184
Accuracy           : 0.8152
Balanced Accuracy  : 0.8094
Sensitivity/Recall : 0.8627
Specificity        : 0.7561
Precision          : 0.8148
F1 Score           : 0.8381
ROC-AUC            : 0.8985
PR-AUC             : 0.9119
Brier Score        : 0.1273

Confusion Matrix:
  TN: 62    FP: 20   
  FN: 14    TP: 88   



## 4. Diagnostic Performance Visualizations
Generating Confusion Matrix, ROC Curve, and Precision-Recall Curve on the held-out test set.


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(splits.y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No CAD', 'CAD Present'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix\n({best_name})')
axes[0].grid(False)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(splits.y_test, y_test_prob)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'ROC-AUC = {test_metrics["roc_auc"]:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].legend(loc='lower right')

# 3. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(splits.y_test, y_test_prob)
baseline_prevalence = np.mean(splits.y_test)
axes[2].plot(rec, prec, color='teal', lw=2, label=f'PR-AUC = {test_metrics["pr_auc"]:.3f}')
axes[2].axhline(baseline_prevalence, color='gray', linestyle='--', label=f'Baseline ({baseline_prevalence:.2f})')
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall (Sensitivity)')
axes[2].set_ylabel('Precision (Positive Predictive Value)')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hd_model_performance.png', dpi=150)
plt.close()
print('Saved diagnostic plot: hd_model_performance.png')


Saved diagnostic plot: hd_model_performance.png


## 5. Sample Size Limitations & Clinical Interpretability
1. **Sample Size Constraints**: The total dataset comprises 920 patients across four historical centers (Cleveland, Hungary, Switzerland, Long Beach). While adequate for prototype benchmarking, prospective clinical deployment requires validation across contemporary multi-center cohorts.
2. **Missing Secondary Tests**: Fluoroscopy (`ca`) and nuclear scans (`thal`) were missing in over 50% of records due to site ordering protocols. The median imputation strategy allows graceful degradation when secondary diagnostics have not been conducted.
3. **Research Prototype Notice**: This model is a research and educational prototype. Model scores reflect statistical correlations and do not constitute clinical diagnostic proof. Emergency clinical safety rules operate independently of model scores.
